This is the online appendix accompanying "When Methods Matter: How Implementation Choices Shape Topic Discovery in Financial Text"

# Requirements

Python: Version ≥ 3.9 is required.

Mallet: A Java-based topic modeling tool, as lda_eval_lib includes a wrapper for Mallet.

To install Mallet, follow the detailed instructions provided at https://programminghistorian.org/en/lessons/topic-modeling-and-mallet

# Text Preprocessing

In [6]:
import lda_eval_lib
import pandas as pd
import os
import spacy
import re
from tqdm import tqdm, tqdm_notebook
tqdm.pandas()

### Loading text

In [2]:
package_path = os.path.dirname(lda_eval_lib.__file__)
data_path = os.path.join(package_path, 'test_text.pkl')
item1a = pd.read_pickle(data_path)

### Removing stopwords 

The following function removes stop words (Loughran and McDonalds available at: https://sraf.nd.edu/loughranmcdonald-master-dictionary/)

Optionally, one can apply stemmer.

In [8]:
# Load stop word lists
package_path = os.path.dirname(lda_eval_lib.__file__)
LMaug_path = os.path.join(package_path, 'StopWord-Augment.txt')
LMlong_path = os.path.join(package_path, 'StopWords_GenericLong.txt')

nlp = spacy.load('en_core_web_sm', disable=['parser'])

# Load the stop word list from the txt file.
with open(LMaug_path) as f:
    stopwords1 = [line.rstrip().lower() for line in f]
with open(LMlong_path) as f:
    stopwords2 = [line.rstrip().lower() for line in f]
stopwords = stopwords1 + stopwords2

def preprocess(doc,stem=False):
    from nltk.stem import PorterStemmer
    ps = PorterStemmer()
    # For each token in the document, we check whether the token (in lower case) is in the stop word list. If it is a
    # stop word, we skip to the next word. If the token is not a stop word, we append it to the "content_no_stopwords"
    # list.
    content_no_stopwords = [token.text for token in nlp(doc) if not token.text.lower() in stopwords]
    
    # Combining the list of tokens into a single string joined with a single character space
    content_no_stopwords = " ".join(content_no_stopwords)
    
    content_no_stopwords = re.sub('[0-9]', ' ', content_no_stopwords)  # remove number characters from the line
    # print(content_string)
    content_no_stopwords = re.sub(r'[^\w\s]', ' ', content_no_stopwords) # replace non(^) word characters or whitespaces with an empty string
    # print(content_string)
    content_no_stopwords = content_no_stopwords.lower() 
    
    wordlist = content_no_stopwords.split()

    if stem==True:
        content_no_stopwords = [ps.stem(word) for word in wordlist if len(word)>1]
    else:
        content_no_stopwords = [word for word in wordlist if len(word)>1]

    # Then return  the list of tokens that are not stopwords
    return content_no_stopwords

item1a['text4LDA'] = item1a['componentText'].progress_apply(lambda row: preprocess(row,stem=True))

100%|█████████████████████████████████████████| 300/300 [07:11<00:00,  1.44s/it]


### (optional) Multiword Expression

In [10]:
import gensim
import gensim.corpora as corpora
from gensim.models import Phrases
from gensim.utils import simple_preprocess
import nltk

data_words = item1a['text4LDA']

# Assuming data_words is already preprocessed and available
bigram = Phrases(data_words, min_count=int(0.01*len(data_words)), threshold=1)
trigram = Phrases(bigram[data_words], min_count=int(0.01*len(data_words)), threshold=1)

nlp = spacy.load('en_core_web_sm', disable=['parser'])

def MWE_POS_CHECKER(MWE):
    # MWE should be the multiple words connected using underscore. e.g., earnings_per_share
    MWE = MWE.replace('_',' ')
    doc = nlp(MWE)
    pos_tag = [token.pos_ for token in doc]

    # Simplify conditions with sets
    allowed_bigrams = [{'NOUN', 'NOUN'}, {'ADJ', 'NOUN'}]
    allowed_trigrams = [{'NOUN', 'NOUN'}, {'ADJ', 'NOUN'}, {'NOUN', 'ADJ'}, {'ADJ', 'ADJ'}]

    if len(pos_tag) == 2:
        return set(pos_tag) in allowed_bigrams
    elif len(pos_tag) == 3:
        return set([pos_tag[0], pos_tag[2]]) in allowed_trigrams
    else:
        return False


# Precompute and filter the bigram vocab
filtered_vocab = {
    word: freq for word, freq in bigram.vocab.items()
    if ((
        b'_' in word and  # Only consider phrases
        (
            freq >= 100 and  # Keep only frequent phrases
            MWE_POS_CHECKER(word.decode('utf-8'))  # Check if POS tagging is valid
        )
    ) | (b'_' not in word))
}

# Replace the old vocab with the filtered one
bigram.vocab = filtered_vocab

# Precompute and filter the trigram vocab
filtered_vocab = {
    word: freq for word, freq in trigram.vocab.items()
    if ((
        b'_' in word and  # Only consider phrases
        (
            freq >= 100 and  # Keep only frequent phrases
            MWE_POS_CHECKER(word.decode('utf-8'))  # Check if POS tagging is valid
        )
    ) | (b'_' not in word))
}

# Replace the old vocab with the filtered one
trigram.vocab = filtered_vocab

bigram_mod = gensim.models.phrases.Phraser(bigram)
trigram_mod = gensim.models.phrases.Phraser(trigram)


def make_bigrams(texts):
    return [bigram_mod[doc] for doc in texts]

def make_trigrams(texts):
    return [trigram_mod[bigram_mod[doc]] for doc in texts]
    
data_bigrams = make_bigrams(data_words)
data_trigrams = make_trigrams(data_words)

tri=[]
for i in trigram.vocab:
    if "_" in i.decode('utf-8'):
        tri.append(i.decode('utf-8'))

# tokenized_text that could alternatively be used for LDA
item1a['text4LDA_MWE'] = data_trigrams

# print a few trigram words
tri[0:10]

['item_risk',
 'trade_price',
 'common_stock',
 'risk_uncertainti',
 'time_time',
 'risk_factor',
 'impact_busi',
 'advers_affect_busi',
 'highli_competit',
 'effect_busi']

In [11]:
item1a

,conm,cik,AR_ID,filingdate,section,componentText,text4LDA,text4LDA_MWE
0,AMERICAN AIRLINES GROUP INC,6201,edgar/data/6201/0000006201-19-000009.txt,2019-02-25,item1a,ITEM 1A. RISK FACTORSBelow are certain risk fa...,"[item, risk, factorsbelow, risk, factor, affec...","[item, risk, factorsbelow, risk, factor, affec..."
1,AMERICAN AIRLINES GROUP INC,6201,edgar/data/6201/0000006201-20-000023.txt,2020-02-19,item1a,ITEM 1A. RISK FACTORSBelow are certain risk fa...,"[item, risk, factorsbelow, risk, factor, affec...","[item, risk, factorsbelow, risk, factor, affec..."
2,AMERICAN AIRLINES GROUP INC,6201,edgar/data/6201/0000006201-21-000014.txt,2021-02-17,item1a,ITEM 1A. RISK FACTORSBelow are certain risk fa...,"[item, risk, factorsbelow, risk, factor, affec...","[item, risk, factorsbelow, risk, factor, affec..."
3,AMERICAN AIRLINES GROUP INC,6201,edgar/data/6201/0000006201-22-000026.txt,2022-02-22,item1a,ITEM 1A. RISK FACTORSBelow are certain risk fa...,"[item, risk, factorsbelow, risk, factor, affec...","[item, risk, factorsbelow, risk, factor, affec..."
4,PINNACLE WEST CAPITAL CORP,764622,edgar/data/764622/0000764622-19-000019.txt,2019-02-22,item1a,ITEM 1A. RISK FACTORS In addition to the facto...,"[item, risk, factor, addit, factor, affect, sp...","[item, risk, factor, addit, factor, affect, sp..."
...,...,...,...,...,...,...,...,...
295,MOLSON COORS BEVERAGE CO,24545,edgar/data/24545/0000024545-22-000005.txt,2022-02-23,item1a,ITEM 1A. RISK FACTORSInvesting in our Company ...,"[item, risk, factorsinvest, involv, risk, inve...","[item, risk, factorsinvest, involv, risk, inve..."
296,CORNING INC,24741,edgar/data/24741/0000024741-19-000016.txt,2019-02-12,item1a,Item 1A. Risk Factors \t\t We operate in rapid...,"[item, risk, factor, oper, rapidli, chang, eco...","[item, risk, factor, oper, rapidli, chang, eco..."
297,CORNING INC,24741,edgar/data/24741/0000024741-20-000014.txt,2020-02-18,item1a,Item 1A. Risk Factors We operate in rapidly ch...,"[item, risk, factor, oper, rapidli, chang, eco...","[item, risk, factor, oper, rapidli, chang, eco..."
298,CORNING INC,24741,edgar/data/24741/0001562762-21-000023.txt,2021-02-12,item1a,Item 1A. Risk Factors We operate in rapidly ch...,"[item, risk, factor, oper, rapidli, chang, eco...","[item, risk, factor, oper, rapidli, chang, eco..."


### LDA run (gensim LDA and modified mallet wrapper)

In [8]:
# Set parameters
minpc = 0.05 
maxpc = 0.5
alpha = "auto"
beta = "auto"
ntopics = 30

# folder to save model and data files
savefolder = '/Users/gitaepark/Downloads'

# path to mallet bin
mallet_path = '/Users/mallet-2.0.8/bin/mallet'

# text for LDA
tokenized_text = item1a['text4LDA'] #one might want to use item1a['text4LDA_MWE']

In [17]:
lda_eval_lib.lda_model.lda_run(tokenized_text, minpc, maxpc, alpha, beta, ntopics, mallet_path, savefolder, gensimpasses = 3, gensimiterations = 3, malletiterations = 100)

 - modelling  gensim_0.05_0.5_30_auto_auto
model saved: /Users/gitaepark/Downloads/gensim_0.05_0.5_30_auto_auto
 - modelling  mallet_0.05_0.5_30_auto_auto
model saved: /Users/gitaepark/Downloads/mallet_0.05_0.5_30_auto_auto


In [14]:
# Load a trained model 
model_path = "/Users/gitaepark/Downloads/mallet_0.05_0.5_30_0.05_0.01"
lda_model, textbeforetokenization, tokenized_text, corpus, id2word = lda_eval_lib.util.load_lda_run(model_path)

# lda_model is other gensim.lda object or gensim.mallet object
lda_model.show_topics(num_topics=-1)

[(0,
  '0.018*"raw" + 0.016*"project" + 0.011*"patents" + 0.010*"venture" + 0.008*"hazardous" + 0.007*"sites" + 0.007*"commodity" + 0.006*"execution" + 0.006*"default" + 0.005*"patent"'),
 (1,
  '0.024*"healthcare" + 0.010*"patents" + 0.010*"agreement" + 0.009*"manufacture" + 0.009*"separation" + 0.008*"patent" + 0.007*"raw" + 0.007*"licenses" + 0.007*"stockholders" + 0.006*"audits"'),
 (2,
  '0.023*"cards" + 0.019*"card" + 0.015*"members" + 0.013*"member" + 0.013*"advertising" + 0.011*"brand" + 0.010*"acceptance" + 0.010*"stations" + 0.009*"travel" + 0.007*"regulators"'),
 (3,
  '0.028*"brands" + 0.020*"water" + 0.019*"consumers" + 0.016*"raw" + 0.013*"beverage" + 0.012*"class" + 0.011*"glass" + 0.011*"retailers" + 0.010*"consumption" + 0.009*"distributors"'),
 (4,
  '0.021*"care" + 0.011*"deterioration" + 0.010*"currencies" + 0.009*"licensing" + 0.008*"medical" + 0.008*"protected" + 0.008*"patient" + 0.007*"interpretations" + 0.006*"taxation" + 0.006*"manufacture"'),
 (5,
  '0.028*"g

### model evaluation

In [15]:
model_path = "/Users/gitaepark/Downloads/mallet_0.05_0.5_30_0.05_0.01"

print(f"LDA model loaded from: {model_path}")

a=lda_eval_lib.metrics.IDXtopics2keep(model_path)
print(f"proportion of valid topics: {a}")

b=lda_eval_lib.metrics.diversity(model_path)
print(f"diversity score: {b}")

c=lda_eval_lib.metrics.granularity(model_path)
print(f"granularity score: {c}")

d=lda_eval_lib.metrics.perplexity(model_path)
print(f"perplexity score: {d}")

e=lda_eval_lib.metrics.coherence(model_path)
print(f"coherence score: {e}")

LDA model loaded from: /Users/gitaepark/Downloads/mallet_0.05_0.5_30_0.05_0.01
proportion of valid topics: 1.0
diversity score: 0.7733333333333333
granularity score: 0.6966666666666667
perplexity score: -7.350240838543099
coherence score: 0.5497179415067865


### GPT assisted tasks (WIT, labelling)

In [2]:
# ChatGPT assisted tasks 
client = lda_eval_lib.util.OpenAI_client()

Enter your API key:  ········


In [ ]:
top_n = 5
bottom_n = 0.15
common_n = 20
chunk_size = 3

# Define the model
gpt_model = "gpt-4o"
model_path = "/Users/gitaepark/Downloads/mallet_0.05_0.5_30_0.05_0.01"

WIT = lda_eval_lib.gpt_wit.word_intrusion_task(model_path, client, top_n, bottom_n, common_n, chunk_size=10,
                           gpt_model="gpt-4o", temperature=0)


In [5]:
WIT.iloc[0]['task']

,Model,Topic,Word0,Word1,Word2,Word3,Word4,Intruder,Response
0,mallet_0.05_0.5_30_0.05_0.01,0,raw,project,patents,venture,hazardous,monetary,hazardous
1,mallet_0.05_0.5_30_0.05_0.01,1,healthcare,patents,agreement,manufacture,separation,expansion,healthcare
2,mallet_0.05_0.5_30_0.05_0.01,2,cards,card,members,member,advertising,lending,lending
3,mallet_0.05_0.5_30_0.05_0.01,3,brands,water,consumers,raw,beverage,senior,senior
4,mallet_0.05_0.5_30_0.05_0.01,4,care,deterioration,currencies,licensing,medical,outcomes,currencies
5,mallet_0.05_0.5_30_0.05_0.01,5,gas,generation,electric,utility,nuclear,strength,strength
6,mallet_0.05_0.5_30_0.05_0.01,6,institutions,institution,participate,title,iv,properly,iv
7,mallet_0.05_0.5_30_0.05_0.01,7,yen,denominated,instruments,rating,derivatives,prospective,prospective
8,mallet_0.05_0.5_30_0.05_0.01,8,oil,gas,drilling,rigs,reserves,partially,partially
9,mallet_0.05_0.5_30_0.05_0.01,9,goods,supplier,locations,located,base,residential,residential


In [6]:
gpt_model = "gpt-4o"
model_path = "/Users/gitaepark/Downloads/mallet_0.05_0.5_30_0.05_0.01"
lda_eval_lib.gpt_labelling.labelling(model_path, client, gpt_model)

Labels checked - overlapping labels identified!
Suggesting new labels...
Done!


,topic,label,rationale,keywords
0,Topic0,Procurement and Project Execution Risks,Emphasizes risks linked with obtaining and han...,"raw, project, patents, venture, hazardous, sit..."
1,Topic1,Healthcare Production and Patenting Risk,Centers on risks related to manufacturing and ...,"healthcare, patents, agreement, manufacture, s..."
2,Topic2,Consumer Payment Systems and Marketing Risks,"The keywords emphasize 'cards', 'card members'...","cards, card, members, member, advertising, bra..."
3,Topic3,Beverage Industry Supply Chain and Consumer Risks,The keywords highlight the beverage industry w...,"brands, water, consumers, raw, beverage, class..."
4,Topic4,Medical Compliance and Currency Risk,Combines risks related to medical compliance a...,"care, deterioration, currencies, licensing, me..."
5,Topic5,Renewable Energy Development Risks,Focuses on risks linked to the development of ...,"gas, generation, electric, utility, nuclear, p..."
6,Topic6,Educational Institution Participation and Fund...,The keywords indicate risks related to educati...,"institutions, institution, participate, title,..."
7,Topic7,Derivative and Hedging Strategy Risks,Emphasizes risks related to the management of ...,"yen, denominated, instruments, rating, derivat..."
8,Topic8,Crude Oil and Gas Extraction Risks,Centers around risks linked to the extraction ...,"oil, gas, drilling, rigs, reserves, exploratio..."
9,Topic9,Component Sourcing and Logistics Vulnerability,Highlights risks arising from sourcing compone...,"goods, supplier, locations, located, base, ski..."


### Visualization

In [ ]:
import pyLDAvis

model_path = "/Users/gitaepark/Downloads/mallet_0.05_0.5_30_auto_auto"

pyldavisdata = lda_eval_lib.custom_pyvis.visualize_topics(model_path)

# Apply pyLDAvis
vis_data= pyLDAvis.prepare(**pyldavisdata, R=30, n_jobs = -1, mds='mmds', sort_topics=False)


![](example_visualization.jpg)